# Pipeline Klasifikasi Aksara Jawa - PyTorch

Pipeline ini mencakup:
1. Data Loading dan Preprocessing
2. Exploratory Data Analysis
3. CNN Pre-trained Model Training (PyTorch + timm)
4. Feature Extraction
5. Ensemble Model (KNN, Logistic Regression, SVC)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
from PIL import Image

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier

from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
print("="*60)
print("GPU/CUDA Configuration Check")
print("="*60)

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
print(f"cuDNN version: {torch.backends.cudnn.version() if torch.cuda.is_available() else 'N/A'}")

if torch.cuda.is_available():
    print(f"\nGPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
    
    device = torch.device('cuda')
    x = torch.randn(100, 100).to(device)
    y = torch.randn(100, 100).to(device)
    z = torch.matmul(x, y)
    print(f"\nTest computation on GPU: SUCCESS")
else:
    print("\nWARNING: No GPU detected!")
    print("Running on CPU - Training will be slower")
    device = torch.device('cpu')

print(f"\nDevice set to: {device}")
print("="*60)

## 1. Data Loading dan Preprocessing

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

class AksaraJawaDataset(Dataset):
    def __init__(self, root_dir, transform=None, preprocess=True):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.preprocess = preprocess
        self.images = []
        self.labels = []
        
        for class_folder in sorted(self.root_dir.iterdir()):
            if class_folder.is_dir():
                class_name = class_folder.name
                for img_path in class_folder.glob('*'):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                        self.images.append(str(img_path))
                        self.labels.append(class_name)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        if self.preprocess:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.equalizeHist(img)
            img = cv2.GaussianBlur(img, (3, 3), 0)
        else:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img = Image.fromarray(img)
        
        if self.transform:
            img = self.transform(img)
        
        return img, label

train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = AksaraJawaDataset('dataset-aksara-jawa/train', transform=train_transform)
val_dataset = AksaraJawaDataset('dataset-aksara-jawa/val', transform=val_transform)

label_encoder = LabelEncoder()
all_labels = train_dataset.labels + val_dataset.labels
label_encoder.fit(all_labels)

num_classes = len(label_encoder.classes_)
print(f"Number of classes: {num_classes}")
print(f"Classes: {label_encoder.classes_}")
print(f"\nTraining samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## 2. Exploratory Data Analysis

In [ ]:
train_counts = pd.Series(train_dataset.labels).value_counts().sort_index()
val_counts = pd.Series(val_dataset.labels).value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(train_counts.index, train_counts.values, color='steelblue')
axes[0].set_title('Training Data Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(val_counts.index, val_counts.values, color='coral')
axes[1].set_title('Validation Data Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTraining data distribution:")
print(train_counts)
print(f"\nMean: {train_counts.mean():.2f}, Std: {train_counts.std():.2f}")

print("\nValidation data distribution:")
print(val_counts)
print(f"\nMean: {val_counts.mean():.2f}, Std: {val_counts.std():.2f}")

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
axes = axes.ravel()

sample_dataset = AksaraJawaDataset('dataset-aksara-jawa/train', transform=None, preprocess=True)

for idx, class_name in enumerate(label_encoder.classes_):
    class_indices = [i for i, label in enumerate(sample_dataset.labels) if label == class_name]
    if class_indices:
        sample_idx = class_indices[0]
        img, _ = sample_dataset[sample_idx]
        axes[idx].imshow(img, cmap='gray')
        axes[idx].set_title(class_name)
        axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 3. CNN Pre-trained Model Training

In [ ]:
class LabelEncodedDataset(Dataset):
    def __init__(self, dataset, label_encoder):
        self.dataset = dataset
        self.label_encoder = label_encoder
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        label_encoded = self.label_encoder.transform([label])[0]
        return img, label_encoded

train_dataset_encoded = LabelEncodedDataset(train_dataset, label_encoder)
val_dataset_encoded = LabelEncodedDataset(val_dataset, label_encoder)

train_loader = DataLoader(train_dataset_encoded, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset_encoded, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
model_names = ['efficientnet_b0', 'resnet50', 'mobilenetv3_large_100', 'vit_tiny_patch16_224']

print("Available models to test:")
for i, name in enumerate(model_names, 1):
    print(f"{i}. {name}")

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def train_model(model_name, num_classes, train_loader, val_loader, device, num_epochs=30, patience=10):
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print('='*60)
    
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    model = model.to(device)
    
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=False)
    
    best_val_acc = 0
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        scheduler.step(val_loss)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} - Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            best_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    model.load_state_dict(best_state)
    print(f"Best validation accuracy: {best_val_acc:.2f}%")
    
    return model, best_val_acc, history

print("Training functions defined")

In [ ]:
models_results = {}
trained_models = {}

for model_name in model_names:
    try:
        model, best_acc, history = train_model(
            model_name, 
            num_classes, 
            train_loader, 
            val_loader, 
            device,
            num_epochs=30,
            patience=10
        )
        models_results[model_name] = {
            'accuracy': best_acc,
            'history': history
        }
        trained_models[model_name] = model
    except Exception as e:
        print(f"Error training {model_name}: {e}")
        continue

print("\n" + "="*60)
print("Model Comparison Results:")
print("="*60)
for model_name, results in models_results.items():
    print(f"{model_name:30s}: {results['accuracy']:.2f}%")
print("="*60)

In [ ]:
best_model_name = max(models_results, key=lambda x: models_results[x]['accuracy'])
best_model = trained_models[best_model_name]
best_accuracy = models_results[best_model_name]['accuracy']
best_history = models_results[best_model_name]['history']

print(f"Best model: {best_model_name}")
print(f"Best validation accuracy: {best_accuracy:.2f}%")

torch.save(best_model.state_dict(), 'best_model.pth')
print("Best model saved to best_model.pth")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (model_name, results) in enumerate(models_results.items()):
    row = idx // 2
    col = idx % 2
    
    if idx < 4:
        history = results['history']
        ax = axes[row, col]
        ax.plot(history['train_acc'], label='Train', linewidth=2)
        ax.plot(history['val_acc'], label='Val', linewidth=2)
        ax.set_title(f'{model_name}\nBest Acc: {results["accuracy"]:.2f}%')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy (%)')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nUsing best model: {best_model_name}")
print(f"Final validation accuracy: {best_accuracy:.2f}%")

In [ ]:
model_accuracies = {name: results['accuracy'] for name, results in models_results.items()}

plt.figure(figsize=(10, 6))
bars = plt.bar(model_accuracies.keys(), model_accuracies.values(), 
               color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
plt.title('Pretrained Models Comparison')
plt.xlabel('Model')
plt.ylabel('Validation Accuracy (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}%',
            ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 4. Feature Extraction

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self, model):
        super(FeatureExtractor, self).__init__()
        self.features = nn.Sequential(*list(model.children())[:-1])
    
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return x

feature_extractor = FeatureExtractor(best_model)
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

print(f"Feature extractor created from {best_model_name}")

In [ ]:
def extract_features(model, data_loader, device):
    features = []
    labels = []
    
    model.eval()
    with torch.no_grad():
        for images, lbls in tqdm(data_loader, desc='Extracting features'):
            images = images.to(device)
            feats = model(images)
            features.append(feats.cpu().numpy())
            labels.append(lbls.numpy())
    
    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)
    return features, labels

print("Extracting features from training data...")
X_train_features, y_train_encoded = extract_features(feature_extractor, train_loader, device)

print("Extracting features from validation data...")
X_val_features, y_val_encoded = extract_features(feature_extractor, val_loader, device)

print(f"\nTraining features shape: {X_train_features.shape}")
print(f"Validation features shape: {X_val_features.shape}")

## 5. Ensemble Model Training

In [ ]:
print("Training KNN Classifier...")
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_features, y_train_encoded)
knn_pred = knn.predict(X_val_features)
knn_accuracy = accuracy_score(y_val_encoded, knn_pred)
print(f"KNN Accuracy: {knn_accuracy:.4f}")

print("\nTraining Logistic Regression...")
lr = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)
lr.fit(X_train_features, y_train_encoded)
lr_pred = lr.predict(X_val_features)
lr_accuracy = accuracy_score(y_val_encoded, lr_pred)
print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")

print("\nTraining SVC...")
svc = SVC(kernel='rbf', probability=True, random_state=42)
svc.fit(X_train_features, y_train_encoded)
svc_pred = svc.predict(X_val_features)
svc_accuracy = accuracy_score(y_val_encoded, svc_pred)
print(f"SVC Accuracy: {svc_accuracy:.4f}")

In [ ]:
print("Training Voting Classifier...")
voting_clf = VotingClassifier(
    estimators=[
        ('knn', knn),
        ('lr', lr),
        ('svc', svc)
    ],
    voting='soft',
    n_jobs=-1
)

voting_clf.fit(X_train_features, y_train_encoded)
voting_pred = voting_clf.predict(X_val_features)
voting_accuracy = accuracy_score(y_val_encoded, voting_pred)

print(f"\nEnsemble Voting Classifier Accuracy: {voting_accuracy:.4f}")

print("\n" + "="*50)
print("Model Comparison:")
print("="*50)
print(f"KNN:                 {knn_accuracy:.4f}")
print(f"Logistic Regression: {lr_accuracy:.4f}")
print(f"SVC:                 {svc_accuracy:.4f}")
print(f"Voting Ensemble:     {voting_accuracy:.4f}")
print("="*50)

## 6. Evaluation

In [ ]:
print("Classification Report for Voting Ensemble:")
print(classification_report(y_val_encoded, voting_pred, 
                          target_names=label_encoder.classes_))

In [ ]:
cm = confusion_matrix(y_val_encoded, voting_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix - Voting Ensemble')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
accuracies = {
    'KNN': knn_accuracy,
    'Logistic Regression': lr_accuracy,
    'SVC': svc_accuracy,
    'Voting Ensemble': voting_accuracy
}

plt.figure(figsize=(10, 6))
bars = plt.bar(accuracies.keys(), accuracies.values(), 
               color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
plt.title('Model Accuracy Comparison')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}',
            ha='center', va='bottom')

plt.tight_layout()
plt.show()